# 04 — Agent Trajectory Evaluation

Evaluate a tool-using agent with `AgentTestCase`:

* Validate that expected tool calls happened.
* Detect loops and step efficiency.
* Judge the final answer with an offline judge.

## 1. Record an agent trajectory

In [ ]:
from checkllm.agents import AgentStep, AgentTestCase, ToolCall

tc = AgentTestCase(
    query="What is the weather in Paris and Berlin?",
    steps=[
        AgentStep(
            action="call_tool",
            tool_call=ToolCall(name="weather", parameters={"city": "Paris"}, result="22C sunny"),
        ),
        AgentStep(
            action="call_tool",
            tool_call=ToolCall(name="weather", parameters={"city": "Berlin"}, result="18C cloudy"),
        ),
        AgentStep(action="respond"),
    ],
    expected_tools=[
        ToolCall(name="weather", parameters={"city": "Paris"}),
        ToolCall(name="weather", parameters={"city": "Berlin"}),
    ],
    final_output="Paris is 22C sunny, Berlin is 18C cloudy.",
)

print(tc.format_trace())

## 2. Validate expected tool calls

In [ ]:
from checkllm.agents import validate_tool_calls

result = validate_tool_calls(tc)
print("tool-call validation:", result.passed, "| score:", result.score)
print("reasoning:", result.reasoning)

## 3. Loop detection and step efficiency

In [ ]:
from collections import Counter

tool_names = [call.name for call in tc.tool_calls]
most_common, count = Counter(tool_names).most_common(1)[0]
print(f"tool {most_common!r} called {count} times")
print("total steps        :", len(tc.steps))
has_loop = count > 5
print("suspected loop     :", has_loop)

## 4. Judge the final answer offline

In [ ]:
from checkllm.judge import JudgeResponse


class FakeJudge:
    """Deterministic in-process judge used to keep this notebook offline.

    Real runs should swap this for ``OpenAIJudge`` / ``AnthropicJudge`` etc.
    """

    def __init__(self, score: float = 0.9, reasoning: str = "looks good") -> None:
        self._score = score
        self._reasoning = reasoning

    async def evaluate(self, prompt: str, system_prompt: str | None = None) -> JudgeResponse:
        return JudgeResponse(
            score=self._score,
            reasoning=self._reasoning,
            raw_output=str(self._score),
            cost=0.0,
        )


judge = FakeJudge(score=0.9, reasoning="offline stub")

In [ ]:
prompt = (
    "Query: " + tc.query + "\n"
    "Answer: " + (tc.final_output or "") + "\n"
    "Is the answer correct and helpful? Score 0-1."
)
resp = await judge.evaluate(prompt=prompt)
print("answer-quality:", resp.score, "|", resp.reasoning)

## 5. Switch to a real provider

In [ ]:
# -------------------------------------------------------------------
# SWITCH TO A REAL PROVIDER
# -------------------------------------------------------------------
# Uncomment and replace the FakeJudge above once you have API keys:
#
# from checkllm.judge import OpenAIJudge
# judge = OpenAIJudge(api_key="sk-...", model="gpt-4o-mini")
#
# from checkllm.judge import AnthropicJudge
# judge = AnthropicJudge(api_key="...", model="claude-3-5-sonnet-20241022")
